# 6주차. AI 기반 환경 스캐닝

## 오늘의 목표

이번 시간에는 어려운 분석 모델을 배우지 않는다. 기획자가 AI와 Python을 활용해 바깥 변화를 빠르게 읽는 방법을 연습한다.

수업이 끝나면 다음을 할 수 있어야 한다.

1. 환경 스캐닝이 무엇인지 설명한다.
2. 뉴스 제목에서 반복되는 키워드를 찾는다.
3. 최근에 증가한 키워드를 약신호 후보로 본다.
4. 분석 결과를 기회, 위험, 다음 행동으로 정리한다.

## 1. 환경 스캐닝이란?

환경 스캐닝은 계획을 세우기 전에 바깥 변화를 살피는 일이다.

예를 들어 청년 주거 정책을 기획한다면 다음 변화를 봐야 한다.

- 월세가 오르고 있는가?
- 청년들이 어떤 지원을 요구하는가?
- 정부나 지자체가 어떤 새 정책을 내는가?
- 기술이나 플랫폼이 문제 해결에 쓰일 수 있는가?

핵심 질문은 하나다.

> 지금은 작아 보이지만, 앞으로 계획에 영향을 줄 변화는 무엇인가?

## 2. 약신호란?

약신호는 아직 크지 않지만 앞으로 중요해질 수 있는 변화의 조짐이다.

쉽게 말해 다음과 같은 신호다.

- 갑자기 자주 보이는 단어
- 전에는 없었는데 최근에 반복되는 이슈
- 작은 사건이지만 여러 곳에서 동시에 나타나는 변화
- 기회일 수도 있고 위험일 수도 있는 변화

기획자는 약신호를 보고 바로 결론을 내리지 않는다. 먼저 "더 확인해야 할 것"으로 정리한다.

## 3. 바이브 코딩 프롬프트

오늘 실습에서 AI에게 요청할 문장이다.

```text
아래 뉴스 제목 목록에서 자주 등장하는 키워드를 세는 Python 코드를 만들어줘.
불용어(the, for, and 같은 의미 없는 단어)는 제외해줘.
결과는 pandas DataFrame으로 보여주고 막대그래프도 그려줘.
코드는 초보자도 이해할 수 있게 짧게 작성해줘.
```

오류가 나면 오류 메시지를 그대로 복사해서 다시 묻는다.

```text
아래 Python 코드에서 오류가 났어. 오류 메시지를 보고 고쳐줘.
[오류 메시지]
```

In [ ]:
# 기본 라이브러리 불러오기
from collections import Counter
import re

import matplotlib.pyplot as plt
import pandas as pd

## 4. 실습 데이터

실제 뉴스 API를 쓰면 API 키와 크롤링 문제가 생긴다. 오늘은 수업용 뉴스 제목 목록을 사용한다.

주제는 `청년 주거`, `AI 행정`, `지역 창업`이 섞여 있다.

In [ ]:
news_titles = [
    "City expands youth housing support program",
    "Rising rent increases demand for public rental housing",
    "Local university opens AI job training course",
    "Small businesses ask city for digital transformation support",
    "Citizens raise privacy concerns over AI welfare screening",
    "New public data platform supports local startup growth",
    "Youth housing applications rise after rent increase",
    "AI tool helps officials review welfare documents faster",
    "Privacy groups call for rules on AI decision systems",
    "City plans more rental housing near transit stations",
    "Startup funding shifts toward AI and public data services",
    "Residents demand privacy standards for welfare AI screening",
]

pd.DataFrame({"title": news_titles})

## 5. 키워드 세기

먼저 뉴스 제목을 단어로 나누고, 의미가 약한 단어를 제거한다.

이 단계는 AI에게 맡겨도 된다. 중요한 것은 코드 암기가 아니라 결과 해석이다.

In [ ]:
stopwords = {
    "for", "and", "the", "on", "over", "near", "more", "new",
    "city", "local", "public", "after", "call", "clearer", "faster", "toward"
}

def tokenize(title):
    words = re.findall(r"[A-Za-z]+", title.lower())
    return [word for word in words if word not in stopwords and (len(word) >= 3 or word == "ai")]

def count_keywords(titles):
    counter = Counter()
    for title in titles:
        counter.update(tokenize(title))
    return (
        pd.DataFrame(counter.items(), columns=["keyword", "count"])
        .sort_values("count", ascending=False)
        .reset_index(drop=True)
    )

keyword_counts = count_keywords(news_titles)
keyword_counts.head(10)

In [ ]:
top = keyword_counts.head(8)

plt.figure(figsize=(8, 4))
plt.bar(top["keyword"], top["count"])
plt.title("Top Keywords in News Titles")
plt.xlabel("Keyword")
plt.ylabel("Count")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

## 6. 약신호 후보 찾기

이번에는 제목 목록을 앞부분과 뒷부분으로 나누어 비교한다.

- 앞부분: earlier news
- 뒷부분: recent news

최근에 언급이 늘어난 키워드는 약신호 후보가 될 수 있다.

In [ ]:
def compare_early_recent(titles):
    midpoint = len(titles) // 2
    early = count_keywords(titles[:midpoint]).rename(columns={"count": "early"})
    recent = count_keywords(titles[midpoint:]).rename(columns={"count": "recent"})

    trend = pd.merge(early, recent, on="keyword", how="outer").fillna(0)
    trend["change"] = trend["recent"] - trend["early"]
    trend["weak_signal"] = trend.apply(
        lambda row: "candidate" if row["change"] >= 1 and row["recent"] >= 2 else "",
        axis=1,
    )
    return trend.sort_values(["change", "recent"], ascending=False).reset_index(drop=True)

trend_table = compare_early_recent(news_titles)
trend_table.head(10)

## 7. 기획 관점으로 해석하기

숫자만 보고 끝내면 분석이다. 기획은 다음 행동까지 생각해야 한다.

다음 표는 키워드를 `기회`, `위험`, `다음 행동`으로 바꾸는 예시다.

In [ ]:
def make_planning_table(trend):
    rows = []
    for _, row in trend.head(6).iterrows():
        keyword = row["keyword"]
        direction = "increasing" if row["change"] > 0 else "stable/decreasing"
        rows.append({
            "keyword": keyword,
            "direction": direction,
            "opportunity": f"Plan a small response related to {keyword}.",
            "risk": f"Check whether {keyword} creates public concern or budget pressure.",
            "next_action": "Collect 5 more recent articles and ask AI to summarize them.",
        })
    return pd.DataFrame(rows)

planning_table = make_planning_table(trend_table)
planning_table

## 8. 학생 실습

관심 있는 정책 또는 산업을 하나 고른다.

예시:

- 청년 주거
- 지역 관광
- 전기차 충전
- 대학 진로 교육
- 고령자 돌봄

AI에게 다음 프롬프트를 입력하고, 나온 코드를 이 노트북에 붙여 넣어 실행한다.

```text
내 관심 주제는 [주제명]이야.
이 주제와 관련된 예시 뉴스 제목 12개를 만들어줘.
그 다음 Python으로 키워드 빈도, 최근 증가 키워드, 약신호 후보를 분석하는 코드를 작성해줘.
마지막에는 기회, 위험, 다음 행동을 표로 정리해줘.
```

In [ ]:
# 여기에 AI가 만들어준 뉴스 제목과 분석 코드를 붙여 넣으세요.

my_news_titles = [
    # "예시 뉴스 제목 1",
    # "예시 뉴스 제목 2",
]

# my_keyword_counts = count_keywords(my_news_titles)
# my_trend_table = compare_early_recent(my_news_titles)
# make_planning_table(my_trend_table)

## 9. 제출물

아래 5문장을 작성하여 제출한다.

1. 내가 고른 주제는 무엇인가?
2. 가장 많이 등장한 키워드는 무엇인가?
3. 최근에 증가한 약신호는 무엇인가?
4. 이것은 기회인가, 위험인가?
5. 기획자는 다음에 무엇을 확인해야 하는가?

## 오늘의 핵심

환경 스캐닝은 정답을 맞히는 일이 아니다. 변화를 빨리 발견하고, 다음에 확인할 질문을 만드는 일이다.